In [2]:
# Install python dependencies if running locally
# (The GitHub CI uses the YAML file to install these, but locally you need them too)
import sys
!{sys.executable} -m pip install scanpy leidenalg python-igraph celltypist pandas matplotlib

import os
import scanpy as sc
import celltypist
import pandas as pd
import matplotlib.pyplot as plt
import warnings

# Suppress warnings for a cleaner notebook output
warnings.filterwarnings('ignore')

# Scanpy settings for high-quality plotting
sc.settings.verbosity = 3
sc.logging.print_header()
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)

print("Environment setup complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 1.7 MB/s  0:00:01m 1.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 3.5 MB/s  0:00:003.5 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 4.7 MB/s  0:00:00m 5.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 8.6 MB/s  0:00:00m 8.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 12.2 MB/s  0:00:00 13.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 12.5 MB/s  0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 11.8 MB/s  0:00:03 eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 12.6 MB/s  0:00:00 13.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 9.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25/25 [celltypist]0m 24/25 [celltypist]py]dels]

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: p

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


Environment setup complete.


In [6]:
import os
import tarfile
import urllib.request
import shutil
import glob
import ssl

# --- 1. Setup Data Directory ---
os.makedirs("data", exist_ok=True)

# --- 2. Download Whitelist (With SSL Fix) ---
url = "https://github.com/f0t1h/3M-february-2018/raw/refs/heads/master/3M-february-2018.txt.gz"
dest = "data/whitelist.txt.gz"

print(f"Downloading whitelist from {url}...")

# Create an unverified SSL context to bypass the macOS certificate error
ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

try:
    with urllib.request.urlopen(url, context=ssl_context) as response, open(dest, 'wb') as out_file:
        shutil.copyfileobj(response, out_file)
    print("Whitelist download successful.")
except Exception as e:
    print(f"Download failed: {e}")

# Unzip the whitelist
if os.path.exists(dest):
    # We use os.system for gunzip as it's often faster/simpler for .gz specifically
    os.system(f"gunzip -f {dest}")
    print("Whitelist extracted.")

# --- 3. Extract the Box Data ---
tar_filename = "toy_read_ref_set.tar.gz"
if os.path.exists(tar_filename):
    print(f"Found {tar_filename}, extracting...")
    with tarfile.open(tar_filename, "r:gz") as tar:
        tar.extractall()
    print("Extraction complete.")
else:
    # Check if files are already extracted (re-run scenario)
    if os.path.exists("genome.fa"):
         print(f"{tar_filename} not found, but genome.fa exists. Continuing...")
    else:
        print(f"WARNING: {tar_filename} not found! Please ensure it is in the folder.")

# --- 4. Auto-Detect and Rename Files ---
# This finds the files wherever they extracted and renames them 
# to the standard names the rest of the notebook expects.

def find_and_rename(extension, target_name):
    # Look for file recursively in current folder
    files = glob.glob(f"**/*{extension}", recursive=True)
    # Filter out files already in the target location or 'data/' cache
    files = [f for f in files if "data/" not in f and f != target_name]
    
    if files:
        print(f"Found {files[0]} -> Renaming to {target_name}")
        shutil.move(files[0], target_name)
    elif os.path.exists(target_name):
        print(f"{target_name} already exists.")
    else:
        print(f"ERROR: Could not find a file ending in {extension}")

# Map extensions to the names the pipeline needs
find_and_rename(".fa", "genome.fa")
find_and_rename(".gtf", "genes.gtf")

# Handle FASTQs (Need to identify read 1 and read 2)
fastqs = sorted(glob.glob("**/*.fastq.gz", recursive=True))
# Filter out files that are already correct
fastqs = [f for f in fastqs if f != "read1.fastq.gz" and f != "read2.fastq.gz"]

if len(fastqs) >= 2:
    print(f"Found read 1: {fastqs[0]} -> Renaming to read1.fastq.gz")
    shutil.move(fastqs[0], "read1.fastq.gz")
    print(f"Found read 2: {fastqs[1]} -> Renaming to read2.fastq.gz")
    shutil.move(fastqs[1], "read2.fastq.gz")
elif os.path.exists("read1.fastq.gz") and os.path.exists("read2.fastq.gz"):
    print("Reads already correctly named.")
else:
    # Fallback: If you are running this multiple times, files might be in 'fastqs' list
    pass

print("\nReady! Your folder should now contain: genome.fa, genes.gtf, read1.fastq.gz, read2.fastq.gz")

Whitelist download successful.
Whitelist extracted.
Found toy_read_ref_set.tar.gz, extracting...
Extraction complete.
Found toy_ref_read/toy_human_ref/fasta/genome.fa -> Renaming to genome.fa
Found toy_ref_read/toy_human_ref/genes/genes.gtf -> Renaming to genes.gtf

Ready! Your folder should now contain: genome.fa, genes.gtf, read1.fastq.gz, read2.fastq.gz


In [7]:
%%bash
# --- Configuration ---
# Inputs (Local files in the same folder)
GENOME="genome.fa"
GTF="genes.gtf"
R1="read1.fastq.gz"
R2="read2.fastq.gz"

# Outputs (Inside the data folder)
IDX="data/salmon_index"
MAP_OUT="data/alevin_out"

# --- Step 1: Generate Transcript-to-Gene (t2g) Map ---
# We need this map to aggregate transcript counts into gene counts
echo "Generating t2g.tsv from GTF..."
grep 'transcript_id' $GTF | \
awk -F';' '{print $1, $3}' | \
sed 's/transcript_id "//' | sed 's/"; gene_id "/\t/' | sed 's/"//' > data/t2g.tsv

# --- Step 2: Build Salmon Index ---
# Creates the reference index for mapping
echo "Building Salmon index (this may take a moment)..."
salmon index -t $GENOME -i $IDX -p 2 --quiet

# --- Step 3: Run Salmon Alevin (Mapping) ---
# Maps the FASTQ reads to the index
# -l ISR: Inward Stranded Reverse (standard for 10x)
# --rad: Outputs a RAD file for alevin-fry
# --sketch: Uses sketch mode for speed
echo "Running Salmon Alevin mapping..."
salmon alevin -l ISR -1 $R1 -2 $R2 \
  --genomeLibDir $IDX \
  --tgMap data/t2g.tsv \
  --output $MAP_OUT \
  --rad --sketch \
  -p 2

Generating t2g.tsv from GTF...
Building Salmon index (this may take a moment)...


bash: line 22: salmon: command not found


Running Salmon Alevin mapping...


bash: line 30: salmon: command not found


CalledProcessError: Command 'b'# --- Configuration ---\n# Inputs (Local files in the same folder)\nGENOME="genome.fa"\nGTF="genes.gtf"\nR1="read1.fastq.gz"\nR2="read2.fastq.gz"\n\n# Outputs (Inside the data folder)\nIDX="data/salmon_index"\nMAP_OUT="data/alevin_out"\n\n# --- Step 1: Generate Transcript-to-Gene (t2g) Map ---\n# We need this map to aggregate transcript counts into gene counts\necho "Generating t2g.tsv from GTF..."\ngrep \'transcript_id\' $GTF | \\\nawk -F\';\' \'{print $1, $3}\' | \\\nsed \'s/transcript_id "//\' | sed \'s/"; gene_id "/\\t/\' | sed \'s/"//\' > data/t2g.tsv\n\n# --- Step 2: Build Salmon Index ---\n# Creates the reference index for mapping\necho "Building Salmon index (this may take a moment)..."\nsalmon index -t $GENOME -i $IDX -p 2 --quiet\n\n# --- Step 3: Run Salmon Alevin (Mapping) ---\n# Maps the FASTQ reads to the index\n# -l ISR: Inward Stranded Reverse (standard for 10x)\n# --rad: Outputs a RAD file for alevin-fry\n# --sketch: Uses sketch mode for s

In [ ]:
%%bash
# --- Configuration ---
MAP_OUT="data/alevin_out"
QUANT_OUT="data/fry_quant"
WHITELIST="data/whitelist.txt"

# --- Step 1: Generate Permit List ---
# Filters barcodes using the downloaded whitelist
echo "Generating permit list..."
alevin-fry generate-permit-list \
  -d forward \
  -i $MAP_OUT \
  -o $QUANT_OUT \
  -u $WHITELIST

# --- Step 2: Collate ---
# Sorts the mapped records
echo "Collating records..."
alevin-fry collate \
  -i $QUANT_OUT \
  -r $MAP_OUT \
  -t 2

# --- Step 3: Quantify ---
# Generates the final cell-gene count matrix
# --use-mtx: Outputs in Matrix Market format (compatible with Scanpy)
echo "Quantifying..."
alevin-fry quant \
  -i $QUANT_OUT \
  -o $QUANT_OUT/res \
  -t 2 \
  -r cr-like \
  -m data/t2g.tsv \
  --use-mtx

In [ ]:
# 1. Load the Data
# We read the matrix generated in the previous cell
print("Loading count matrix...")
adata = sc.read_10x_mtx(
    'data/fry_quant/res/alevin',
    var_names='gene_symbols', 
    cache=True
)

# 2. Quality Control (QC)
# Identify mitochondrial genes (start with MT-)
adata.var['mt'] = adata.var_names.str.startswith('MT-') 
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

print(f"Original cell count: {adata.n_obs}")

# Filter low quality cells/genes
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

# Filter cells with high mitochondrial content (indicates stress/death)
# We use < 5% as a standard threshold
adata = adata[adata.obs.pct_counts_mt < 5, :]

print(f"Filtered cell count: {adata.n_obs}")

# 3. Normalization & Log Transformation
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# 4. Feature Selection
# Select highly variable genes for clustering
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
adata = adata[:, adata.var.highly_variable]

# 5. Scaling & Dimensionality Reduction
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)

# 6. Clustering (Leiden Algorithm)
sc.tl.leiden(adata)

# 7. Plot the Clustering Result
sc.pl.umap(adata, color=['leiden'], title="Leiden Clustering")

In [ ]:
# 1. Load/Download CellTypist Model
# 'Immune_All_Low' is a robust general model for immune cells.
# It will download automatically if not present.
print("Loading CellTypist model...")
model = celltypist.models.Model.load(model='Immune_All_Low.pkl')

# 2. Run Prediction
# majority_voting=True refines the predictions based on the cell's neighbors
print("Annotating cells...")
predictions = celltypist.annotate(adata, model='Immune_All_Low.pkl', majority_voting=True)

# 3. Assign Labels to AnnData
adata.obs['cell_type'] = predictions.predicted_labels['predicted_labels']
adata.obs['conf_score'] = predictions.predicted_labels['conf_score']

# 4. Visualization
# Plot UMAP with cell type labels
sc.pl.umap(adata, color=['cell_type'], title="CellTypist Annotation", legend_loc='on data')

# 5. Summary
print("\nDetected Cell Types:")
print(adata.obs['cell_type'].value_counts())